In [1]:
import pandas as pd

df = pd.read_csv("data/Sample - Superstore.csv", encoding="latin1")

### Insight
This step loads the retail dataset and creates the foundation for the analysis.

In [2]:
df.head()

df.info()

df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


### Insight
A quick preview shows the table shape, data types, and the range of values in the dataset.

In [3]:
import sqlite3

conn = sqlite3.connect("superstore.db")
cursor = conn.cursor()

### Insight
A local database is being set up so the analysis can use SQL queries more naturally.

In [4]:
df.to_sql(
    "superstore_raw",
    conn,
    if_exists="replace",
    index=False
)

9994

### Insight
The raw data is now stored in a structured table, making it easier to query and transform.

In [5]:
cursor.execute("""
CREATE TABLE customers AS
SELECT DISTINCT
    "Customer ID",
    "Customer Name",
    Segment,
    Country,
    State,
    City
FROM superstore_raw;
""")

conn.commit()

### Insight
Customer information is separated into its own table so it can be analyzed more cleanly.

### Insight
Product data is now organized into a dedicated table for clearer analysis.

### Insight
Order-level data is separated so sales, profit, and quantity can be studied directly.

### Insight
These counts confirm that the cleaned tables contain the expected business data.

### Insight
This confirms the number of unique customers in the source data.

### Insight
The customer table is being refreshed so each customer appears only once.

In [6]:
cursor.execute("""
DROP TABLE IF EXISTS products;
""")

cursor.execute("""
CREATE TABLE products AS
SELECT DISTINCT
    "Product ID",
    Category,
    "Sub-Category",
    "Product Name"
FROM superstore_raw;
""")

conn.commit()

In [7]:
cursor.execute("""
DROP TABLE IF EXISTS orders;
""")

cursor.execute("""
CREATE TABLE orders AS
SELECT
    "Order ID",
    "Order Date",
    "Ship Date",
    "Ship Mode",
    "Customer ID",
    "Product ID",
    Sales,
    Quantity,
    Discount,
    Profit
FROM superstore_raw;
""")

conn.commit()

In [8]:
print(pd.read_sql("SELECT COUNT(*) AS Customers FROM customers;", conn))

print(pd.read_sql("SELECT COUNT(*) AS Products FROM products;", conn))

print(pd.read_sql("SELECT COUNT(*) AS Orders FROM orders;", conn))

   Customers
0       4688
   Products
0      1894
   Orders
0    9994


In [9]:
pd.read_sql("""
SELECT COUNT(DISTINCT "Customer ID") AS Unique_Customers
FROM superstore_raw;
""", conn)

,Unique_Customers
0,793


In [10]:
cursor.execute("DROP TABLE IF EXISTS customers;")
conn.commit()

### Insight
This step standardizes customer records to avoid duplicate customer entries.

### Insight
The updated customer table now has the expected number of unique customers.

### Insight
This query identifies orders that are above the average sales level.

### Insight
This highlights the highest-value order for each customer.

### Insight
This view ranks customers by total sales to show who contributes most to revenue.

In [11]:
cursor.execute("""
CREATE TABLE customers AS
SELECT
    "Customer ID",
    MIN("Customer Name") AS "Customer Name",
    MIN(Segment) AS Segment,
    MIN(Country) AS Country,
    MIN(State) AS State,
    MIN(City) AS City
FROM superstore_raw
GROUP BY "Customer ID";
""")

conn.commit()

In [12]:
pd.read_sql("SELECT COUNT(*) AS Customers FROM customers;", conn)

,Customers
0,793


In [13]:
query = """
SELECT *
FROM orders
WHERE Sales > (
    SELECT AVG(Sales)
    FROM orders
);
"""

result = pd.read_sql(query, conn)
result.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,TEC-PH-10002275,907.1520,6,0.20,90.7152
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092


In [14]:
query = """
SELECT *
FROM orders o
WHERE Sales = (
    SELECT MAX(Sales)
    FROM orders
    WHERE "Customer ID" = o."Customer ID"
);
"""

pd.read_sql(query, conn)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
1,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
2,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
3,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,FUR-TA-10000577,1044.6300,3,0.00,240.2649
4,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,FUR-BO-10004834,3083.4300,7,0.50,-1665.0522
...,...,...,...,...,...,...,...,...,...,...
790,CA-2015-159534,3/20/2015,3/23/2015,First Class,DH-13075,OFF-BI-10003656,1087.9360,8,0.20,353.5792
791,CA-2016-129630,9/4/2016,9/4/2016,Same Day,IM-15055,TEC-CO-10003763,2799.9600,5,0.20,944.9865
792,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,OFF-AP-10002945,2405.2000,8,0.00,793.7160
793,CA-2017-153871,12/11/2017,12/17/2017,Standard Class,RB-19435,OFF-BI-10004600,735.9800,2,0.00,331.1910


In [15]:
query = """
WITH CustomerSales AS (
    SELECT
        "Customer ID",
        SUM(Sales) AS TotalSales
    FROM orders
    GROUP BY "Customer ID"
)

SELECT *
FROM CustomerSales
ORDER BY TotalSales DESC;
"""

pd.read_sql(query, conn)

,Customer ID,TotalSales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
788,RS-19870,22.328
789,MG-18205,16.739
790,CJ-11875,16.520
791,LD-16855,5.304


### Insight
This query assigns a rank to each order within a customer’s sales history.

### Insight
This query ranks customers by total sales in a clearer, business-friendly way.

### Insight
This combines customer and order data to analyze revenue by customer profile.

### Insight
The top customers section highlights the strongest revenue contributors.

### Insight
This surfaces the products that bring in the most sales revenue.

In [16]:
query = """
SELECT
    "Customer ID",
    Sales,
    ROW_NUMBER() OVER (
        PARTITION BY "Customer ID"
        ORDER BY Sales DESC
    ) AS RowNum
FROM orders;
"""

pd.read_sql(query, conn)

,Customer ID,Sales,RowNum
0,AA-10315,3930.072,1
1,AA-10315,673.568,2
2,AA-10315,431.976,3
3,AA-10315,362.940,4
4,AA-10315,52.980,5
...,...,...,...
9989,ZD-21925,61.440,5
9990,ZD-21925,22.720,6
9991,ZD-21925,16.720,7
9992,ZD-21925,15.984,8


In [17]:
query = """
WITH CustomerSales AS (

SELECT
    "Customer ID",
    SUM(Sales) AS TotalSales

FROM orders

GROUP BY "Customer ID"

)

SELECT
    *,
    RANK() OVER (
        ORDER BY TotalSales DESC
    ) AS CustomerRank

FROM CustomerSales;
"""

pd.read_sql(query, conn)

,Customer ID,TotalSales,CustomerRank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


In [18]:
query = """
WITH CustomerSales AS (

SELECT

c."Customer ID",

c."Customer Name",

SUM(o.Sales) AS TotalSales

FROM customers c

JOIN orders o

ON c."Customer ID" = o."Customer ID"

GROUP BY

c."Customer ID",

c."Customer Name"

)

SELECT
*,
RANK() OVER (
ORDER BY TotalSales DESC
) AS Rank

FROM CustomerSales;
"""

pd.read_sql(query, conn)

,Customer ID,Customer Name,TotalSales,Rank
0,SM-20320,Sean Miller,25043.050,1
1,TC-20980,Tamara Chand,19052.218,2
2,RB-19360,Raymond Buch,15117.339,3
3,TA-21385,Tom Ashbrook,14595.620,4
4,AB-10105,Adrian Barton,14473.571,5
...,...,...,...,...
788,RS-19870,Roy Skaria,22.328,789
789,MG-18205,Mitch Gastineau,16.739,790
790,CJ-11875,Carl Jackson,16.520,791
791,LD-16855,Lela Donovan,5.304,792


In [19]:
query = """
SELECT
    c."Customer ID",
    c."Customer Name",
    ROUND(SUM(o.Sales),2) AS TotalSales

FROM customers c

JOIN orders o

ON c."Customer ID" = o."Customer ID"

GROUP BY
    c."Customer ID",
    c."Customer Name"

ORDER BY TotalSales DESC

LIMIT 10;
"""

top_customers = pd.read_sql(query, conn)

top_customers

,Customer ID,Customer Name,TotalSales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.23
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.30
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


In [20]:
query = """
SELECT

p."Product Name",

ROUND(SUM(o.Sales),2) AS TotalSales

FROM products p

JOIN orders o

ON p."Product ID" = o."Product ID"

GROUP BY

p."Product Name"

ORDER BY TotalSales DESC

LIMIT 10;
"""

top_products = pd.read_sql(query, conn)

top_products

,Product Name,TotalSales
0,Canon imageCLASS 2200 Advanced Copier,61599.82
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.58
4,GBC DocuBind TL300 Electric Binding System,19823.48
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50
6,Hewlett Packard LaserJet 3310 Copier,18839.69
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90
8,GBC DocuBind P400 Electric Binding System,17965.07
9,High Speed Automatic Electric Letter Opener,17030.31


### Insight
This query helps identify orders that may be hurting overall profitability.

### Insight
This comparison shows which categories are driving the most sales volume.

### Insight
This highlights customers who may need extra engagement to become repeat buyers.

### Insight
This final query focuses on high-value orders that stand out from the average sales level.

In [21]:
query = """
SELECT

"Order ID",

"Customer ID",

Sales,

Profit

FROM orders

WHERE Profit < 0

ORDER BY Profit ASC;
"""

loss_orders = pd.read_sql(query, conn)

loss_orders.head(20)

,Order ID,Customer ID,Sales,Profit
0,CA-2016-108196,CS-12505,4499.985,-6599.9780
1,US-2017-168116,GT-14635,7999.980,-3839.9904
2,CA-2014-169019,LF-17185,2177.584,-3701.8928
3,CA-2017-134845,SR-20425,2549.985,-3399.9800
4,US-2017-122714,HG-14965,1889.990,-2929.4845
5,CA-2015-147830,NF-18385,1799.994,-2639.9912
6,CA-2017-131254,NC-18415,1525.188,-2287.7820
7,CA-2015-116638,JH-15985,4297.644,-1862.3124
8,CA-2016-130946,ZC-21910,1088.792,-1850.9464
9,CA-2014-145317,SM-20320,22638.480,-1811.0784


In [22]:
query = """
SELECT

p.Category,

ROUND(SUM(o.Sales),2) AS TotalSales

FROM products p

JOIN orders o

ON p."Product ID" = o."Product ID"

GROUP BY

p.Category

ORDER BY TotalSales DESC;
"""

category_sales = pd.read_sql(query, conn)

category_sales

,Category,TotalSales
0,Technology,893633.28
1,Furniture,764284.65
2,Office Supplies,736748.59


In [23]:
query = """
SELECT
    c."Customer ID",
    c."Customer Name",
    COUNT(DISTINCT o."Order ID") AS TotalOrders
FROM customers c
JOIN orders o
ON c."Customer ID" = o."Customer ID"
GROUP BY
    c."Customer ID",
    c."Customer Name"
HAVING COUNT(DISTINCT o."Order ID") = 1;
"""

single_order_customers = pd.read_sql(query, conn)

single_order_customers

,Customer ID,Customer Name,TotalOrders
0,AO-10810,Anthony O'Donnell,1
1,AR-10570,Anemone Ratner,1
2,CJ-11875,Carl Jackson,1
3,JC-15385,Jenna Caffey,1
4,JR-15700,Jocasta Rupert,1
5,LD-16855,Lela Donovan,1
6,MG-18205,Mitch Gastineau,1
7,PH-18790,Patricia Hirasaki,1
8,RE-19405,Ricardo Emerson,1
9,RM-19750,Roland Murray,1


In [24]:
query = """
SELECT
    "Order ID",
    "Customer ID",
    Sales,
    Profit
FROM orders
WHERE Sales > (
    SELECT AVG(Sales)
    FROM orders
)
ORDER BY Sales DESC;
"""

above_avg_sales = pd.read_sql(query, conn)

above_avg_sales

,Order ID,Customer ID,Sales,Profit
0,CA-2014-145317,SM-20320,22638.480,-1811.0784
1,CA-2016-118689,TC-20980,17499.950,8399.9760
2,CA-2017-140151,RB-19360,13999.960,6719.9808
3,CA-2017-127180,TA-21385,11199.968,3919.9888
4,CA-2017-166709,HL-15040,10499.970,5039.9856
...,...,...,...,...
2355,CA-2014-147298,AG-10300,230.280,23.0280
2356,CA-2017-161956,DR-12880,230.280,23.0280
2357,US-2014-106334,JF-15490,230.280,23.0280
2358,US-2016-168095,MC-17425,230.280,23.0280
